In [4]:
import pandas as pd
import numpy as np
import time

# ============================================
# FAST EDA FOR 10M+ RECORDS
# ============================================

print("🚀 FAST EDA PIPELINE FOR LARGE DATASETS\n")

# ============================================
# STEP 1: LOAD EFFICIENTLY (SAMPLE-BASED FOR SPEED)
# ============================================

start = time.time()

# Load train.csv with dtype optimization
train = pd.read_csv(
    '/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/train.csv',
    parse_dates=['date'],
    dtype={
        'store_nbr': 'uint16',      # uint16 for store numbers (max ~65k)
        'sales': 'float32',         # float32 instead of float64
        'onpromotion': 'uint8'      # 0 or 1, use uint8
    }
)

load_time = time.time() - start
print(f"✓ Train data loaded in {load_time:.2f}s")
print(f"  Shape: {train.shape}")
print(f"  Memory: {train.memory_usage(deep=True).sum() / 1024**2:.1f} MB\n")

# ============================================
# STEP 2: FAST COLUMN INSPECTION
# ============================================

print("="*60)
print("DATASET STRUCTURE (FAST CHECK)")
print("="*60)
print(f"Columns: {train.columns.tolist()}")
print(f"Data types:\n{train.dtypes}\n")

# ============================================
# STEP 3: QUICK STATISTICS (NO FULL SCAN)
# ============================================

print("="*60)
print("QUICK STATISTICS")
print("="*60)

print(f"\nDate range: {train['date'].min()} to {train['date'].max()}")
print(f"Total days: {(train['date'].max() - train['date'].min()).days}")

print(f"\nStores: {train['store_nbr'].nunique()} unique")
print(f"Sales - Mean: {train['sales'].mean():.2f}, Median: {train['sales'].median():.2f}")
print(f"Zero sales records: {(train['sales'] == 0).sum()} ({(train['sales'] == 0).mean()*100:.1f}%)")
print(f"On promotion records: {train['onpromotion'].sum()} ({train['onpromotion'].mean()*100:.1f}%)\n")

# ============================================
# STEP 4: SAMPLE-BASED ANALYSIS (FOR SPEED)
# ============================================

print("="*60)
print("SAMPLE-BASED ANALYSIS (Stratified by store)")
print("="*60)

# Take a stratified sample - fast and representative
sample_size = 100_000  # 100k rows (1% of 10M)
sample_df = train.groupby('store_nbr', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), max(1, int(len(x) * sample_size / len(train)))), random_state=42)
)

print(f"\nSample size: {len(sample_df):,} rows (representative)")
print(f"Sample statistics:")
print(sample_df[['sales', 'onpromotion']].describe())

# ============================================
# STEP 5: FAST GROUPBY AGGREGATIONS
# ============================================

print("\n" + "="*60)
print("AGGREGATE STATISTICS (Fast groupby)")
print("="*60)

# Use categorical to speed up groupby
train['store_nbr'] = train['store_nbr'].astype('category')

print("\nSales by store (top 10):")
top_stores = train.groupby('store_nbr', observed=True)['sales'].agg(['sum', 'mean', 'count']).nlargest(10, 'sum')
print(top_stores)

# ============================================
# STEP 6: LOAD OTHER METADATA (SMALL FILES)
# ============================================

print("\n" + "="*60)
print("LOAD METADATA FILES (Should be small)")
print("="*60)

try:
    stores = pd.read_csv('/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/stores.csv')
    print(f"✓ Stores: {stores.shape}")
    print(stores.head(3))
except FileNotFoundError:
    print("✗ stores.csv not found")

try:
    items = pd.read_csv('/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/items.csv')
    print(f"\n✓ Items: {items.shape}")
    print(items.head(3))
except FileNotFoundError:
    print("✗ items.csv not found")

try:
    holidays = pd.read_csv('/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/holidays_events.csv', parse_dates=['date'])
    print(f"\n✓ Holidays: {holidays.shape}")
    print(holidays.head(3))
except FileNotFoundError:
    print("✗ holidays_events.csv not found")

# ============================================
# STEP 7: TIME SERIES SAMPLING FOR VIZ
# ============================================

print("\n" + "="*60)
print("TIME SERIES SAMPLE (For visualization)")
print("="*60)

# Aggregate by date only (much smaller)
daily_sales = train.groupby('date')['sales'].agg(['sum', 'mean', 'count']).reset_index()
print(f"\nDaily sales aggregated: {len(daily_sales)} rows")
print(daily_sales.head(10))

# ============================================
# STEP 8: DATA QUALITY CHECKS (Optimized)
# ============================================

print("\n" + "="*60)
print("DATA QUALITY CHECKS (Fast)")
print("="*60)

print(f"\nNull values: {train.isnull().sum().sum()}")
print(f"Negative sales: {(train['sales'] < 0).sum()}")
print(f"Date coverage: {train['date'].nunique()} unique dates")

# ============================================
# STEP 9: KEY INSIGHTS FOR MODELING
# ============================================

print("\n" + "="*60)
print("KEY INSIGHTS FOR YOUR PROJECT")
print("="*60)

print(f"""
✓ Time Series: {len(daily_sales)} daily observations
✓ Stores: {train['store_nbr'].nunique()} unique stores
✓ Date range: {(train['date'].max() - train['date'].min()).days} days
✓ Sparsity: {(train['sales'] == 0).mean()*100:.1f}% zero sales (handle this!)
✓ Promotions: {train['onpromotion'].mean()*100:.1f}% items on promotion

NEXT STEPS:
1. Merge with stores.csv for demographics
2. Merge with holidays.csv for seasonality features
3. Create lagged features from time series
4. Filter to 1-2 stores for initial modeling
5. Build demand forecast model
""")

print("\n✓ EDA COMPLETE!")
print(f"Total execution time: {time.time() - start:.2f}s")


🚀 FAST EDA PIPELINE FOR LARGE DATASETS



ValueError: Integer column has NA values in column 5

In [1]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

print("🚀 STREAMING EDA FOR MASSIVE DATASETS\n")

# ============================================
# STRATEGY: USE CHUNKS + AGGREGATION
# ============================================

csv_path = '/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/train.csv'

# Check file size first
file_size_gb = Path(csv_path).stat().st_size / (1024**3)
print(f"File size: {file_size_gb:.2f} GB")
print(f"⚠️ Too large for full load. Using chunked streaming...\n")

# ============================================
# STEP 1: GET COLUMN INFO (Read only 1000 rows)
# ============================================

print("="*60)
print("STEP 1: INSPECT COLUMNS")
print("="*60)

sample = pd.read_csv(csv_path, nrows=1000, low_memory=False)
print(f"\nColumns: {sample.columns.tolist()}")
print(f"Data types:")
print(sample.dtypes)
print(f"\nFirst 3 rows:")
print(sample.head(3))

# ============================================
# STEP 2: STREAMING AGGREGATION
# ============================================

print("\n" + "="*60)
print("STEP 2: STREAMING AGGREGATION (Process in chunks)")
print("="*60)

chunk_size = 100_000  # Process 100k rows at a time
chunks_processed = 0

# Initialize accumulators
total_records = 0
total_sales = 0
total_on_promo = 0
min_sales = float('inf')
max_sales = float('-inf')
stores_seen = set()
dates_seen = set()
zero_sales_count = 0

start = time.time()

# Read in chunks
for chunk in pd.read_csv(
    csv_path,
    chunksize=chunk_size,
    low_memory=False,
    dtype={'store_nbr': 'uint16', 'sales': 'float32'}  # Only type critical columns
):
    
    # Convert boolean string
    chunk['onpromotion'] = chunk['onpromotion'].astype(str).map({'True': 1, 'False': 0})
    
    # Accumulate stats
    total_records += len(chunk)
    total_sales += chunk['sales'].sum()
    total_on_promo += chunk['onpromotion'].sum()
    min_sales = min(min_sales, chunk['sales'].min())
    max_sales = max(max_sales, chunk['sales'].max())
    zero_sales_count += (chunk['sales'] == 0).sum()
    
    stores_seen.update(chunk['store_nbr'].unique())
    dates_seen.update(pd.to_datetime(chunk['date']).unique())
    
    chunks_processed += 1
    if chunks_processed % 10 == 0:
        print(f"  ✓ Processed {chunks_processed * chunk_size:,} records ({(time.time()-start):.1f}s)")

load_time = time.time() - start

# ============================================
# DISPLAY RESULTS
# ============================================

print("\n" + "="*60)
print("DATASET STATISTICS (From streaming)")
print("="*60)

print(f"\nRecords: {total_records:,}")
print(f"Stores: {len(stores_seen)} unique")
print(f"Date range: {min(dates_seen)} to {max(dates_seen)}")
print(f"Total days: {(max(dates_seen) - min(dates_seen)).days}")

print(f"\nSales stats:")
print(f"  Total: {total_sales:,.0f}")
print(f"  Average per record: {total_sales/total_records:.2f}")
print(f"  Min: {min_sales:.2f}")
print(f"  Max: {max_sales:.2f}")
print(f"  Zero sales: {zero_sales_count:,} ({zero_sales_count/total_records*100:.1f}%)")

print(f"\nPromotion:")
print(f"  On promo: {total_on_promo:,} ({total_on_promo/total_records*100:.1f}%)")

print(f"\n✓ Streaming EDA complete in {load_time:.1f}s")
print(f"  Chunks processed: {chunks_processed}")

# ============================================
# STEP 3: LOAD ONLY WHAT YOU NEED (Subset)
# ============================================

print("\n" + "="*60)
print("STEP 3: LOAD SUBSET FOR DETAILED ANALYSIS")
print("="*60)

# Load just first store for analysis
target_store = min(stores_seen)
print(f"\nLoading data for store {target_store} only...")

df_subset = pd.read_csv(
    csv_path,
    usecols=['date', 'store_nbr', 'sales', 'onpromotion'],  # Only needed columns
    dtype={'store_nbr': 'uint16', 'sales': 'float32'}
)

# Filter to single store
df_store = df_subset[df_subset['store_nbr'] == target_store].copy()
df_store['onpromotion'] = df_store['onpromotion'].astype(str).map({'True': 1, 'False': 0})
df_store['date'] = pd.to_datetime(df_store['date'])

print(f"Store {target_store} records: {len(df_store):,}")
print(f"Date range: {df_store['date'].min()} to {df_store['date'].max()}")
print(f"\nSales stats for store:")
print(df_store['sales'].describe())

# ============================================
# STEP 4: RECOMMENDATION FOR MODELING
# ============================================

print("\n" + "="*60)
print("RECOMMENDATION FOR YOUR PROJECT")
print("="*60)

print(f"""
✓ Total dataset too large: {file_size_gb:.2f} GB

APPROACH OPTIONS:

1. SAMPLE APPROACH (RECOMMENDED):
   - Use pd.read_csv(..., nrows=1_000_000) 
   - Get representative 10% of data
   - Build model on sample (~1-2 GB)
   - Validate on different subset

2. STORE-LEVEL APPROACH (BEST FOR LEARNING):
   - Pick 2-3 high-volume stores
   - Load those stores only
   - Build store-specific models
   - Demonstrate methodology

3. DASK/POLARS APPROACH (PRODUCTION):
   - Use Dask for distributed processing
   - Or use Polars (faster than pandas)
   - Process in parallel chunks

4. DOWNSAMPLING:
   - Use every Nth row: pd.read_csv(..., skiprows=range(1, 9))
   - Aggregate to weekly instead of daily
   - Sample stratified by store

SUGGESTED: Use STORE-LEVEL approach for your thesis
- Pick stores 1, 10, 20 (different tiers)
- Load 30 days each
- Show full pipeline on manageable data
- Document scalability to full dataset
""")

print(f"\nTotal execution time: {load_time:.1f}s (FAST!)")


/Users/shashwatkushwaha/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


🚀 STREAMING EDA FOR MASSIVE DATASETS

File size: 4.65 GB
⚠️ Too large for full load. Using chunked streaming...

STEP 1: INSPECT COLUMNS

Columns: ['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion']
Data types:
id               int64
date            object
store_nbr        int64
item_nbr         int64
unit_sales     float64
onpromotion    float64
dtype: object

First 3 rows:
   id        date  store_nbr  item_nbr  unit_sales  onpromotion
0   0  2013-01-01         25    103665         7.0          NaN
1   1  2013-01-01         25    105574         1.0          NaN
2   2  2013-01-01         25    105575         2.0          NaN

STEP 2: STREAMING AGGREGATION (Process in chunks)


KeyError: 'sales'

In [2]:
import pandas as pd

csv_path = '/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/train.csv'

# Read just first 10 rows
sample = pd.read_csv(csv_path, nrows=10)

print("EXACT COLUMN NAMES:")
print(sample.columns.tolist())
print("\nFIRST 3 ROWS:")
print(sample.head(3))

EXACT COLUMN NAMES:
['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion']

FIRST 3 ROWS:
   id        date  store_nbr  item_nbr  unit_sales  onpromotion
0   0  2013-01-01         25    103665         7.0          NaN
1   1  2013-01-01         25    105574         1.0          NaN
2   2  2013-01-01         25    105575         2.0          NaN


In [ ]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

print("🚀 OPTIMIZED STREAMING EDA - CORPORACIÓN FAVORITA\n")

csv_path = '/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/train.csv'

# ============================================
# CHECK FILE SIZE
# ============================================

file_size_gb = Path(csv_path).stat().st_size / (1024**3)
print(f"File size: {file_size_gb:.2f} GB")
print(f"Using optimized chunked streaming...\n")

# ============================================
# STEP 1: DEFINE COLUMN MAPPING (YOUR ACTUAL COLUMNS)
# ============================================

col_mapping = {
    'date_col': 'date',
    'store_col': 'store_nbr',
    'item_col': 'item_nbr',
    'demand_col': 'unit_sales',
    'promo_col': 'onpromotion',
    'id_col': 'id'
}

# ============================================
# STEP 2: STREAMING AGGREGATION (NO FULL LOAD)
# ============================================

print("="*60)
print("STREAMING ANALYSIS (Processing in chunks)")
print("="*60)

chunk_size = 100_000  # Process 100k rows at a time
chunks_processed = 0

# Initialize accumulators (low memory footprint)
total_records = 0
total_unit_sales = 0
total_on_promo = 0
min_sales = float('inf')
max_sales = float('-inf')
stores_seen = set()
items_seen = set()
dates_seen = set()
zero_sales_count = 0

start = time.time()

print(f"\nColumns to process:")
for key, val in col_mapping.items():
    print(f"  {key}: {val}")

print(f"\nProcessing chunks of {chunk_size:,} rows...\n")

# Read in chunks - ONLY load needed columns for speed
needed_cols = [col_mapping['date_col'], col_mapping['store_col'], 
               col_mapping['item_col'], col_mapping['demand_col'], col_mapping['promo_col']]

for chunk in pd.read_csv(
    csv_path,
    chunksize=chunk_size,
    usecols=needed_cols,  # Only load needed columns
    dtype={
        col_mapping['store_col']: 'uint16',
        col_mapping['item_col']: 'uint32',
        col_mapping['demand_col']: 'float32',
        col_mapping['promo_col']: 'float32'  # NaN-compatible
    },
    low_memory=True
):
    
    # Accumulate stats
    total_records += len(chunk)
    
    # Unit sales stats
    total_unit_sales += chunk[col_mapping['demand_col']].sum()
    min_sales = min(min_sales, chunk[col_mapping['demand_col']].min())
    max_sales = max(max_sales, chunk[col_mapping['demand_col']].max())
    zero_sales_count += (chunk[col_mapping['demand_col']] == 0).sum()
    
    # Promotion stats (handle NaN)
    promo_valid = chunk[col_mapping['promo_col']].dropna()
    total_on_promo += (promo_valid == 1.0).sum() if len(promo_valid) > 0 else 0
    
    # Unique values
    stores_seen.update(chunk[col_mapping['store_col']].unique())
    items_seen.update(chunk[col_mapping['item_col']].unique())
    
    # Parse dates (efficient)
    try:
        dates = pd.to_datetime(chunk[col_mapping['date_col']], format='%Y-%m-%d')
        dates_seen.update(dates.unique())
    except:
        dates_seen.update(chunk[col_mapping['date_col']].unique())
    
    chunks_processed += 1
    if chunks_processed % 10 == 0:
        elapsed = time.time() - start
        rate = (chunks_processed * chunk_size) / elapsed
        print(f"  ✓ {chunks_processed * chunk_size:,} records ({elapsed:.1f}s, {rate:.0f} rows/sec)")

load_time = time.time() - start

# ============================================
# STEP 3: DISPLAY RESULTS
# ============================================

print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)

print(f"\n📊 SCALE:")
print(f"  Total records: {total_records:,}")
print(f"  Stores: {len(stores_seen)} unique")
print(f"  Items/Products: {len(items_seen):,} unique")

if dates_seen:
    min_date = min(dates_seen)
    max_date = max(dates_seen)
    print(f"  Date range: {min_date} to {max_date}")
    print(f"  Total days: {(max_date - min_date).days}")

print(f"\n📈 UNIT SALES STATS:")
print(f"  Total: {total_unit_sales:,.0f} units")
print(f"  Average per record: {total_unit_sales/total_records:.2f} units")
print(f"  Min: {min_sales:.2f}")
print(f"  Max: {max_sales:.2f}")
print(f"  Zero sales: {zero_sales_count:,} ({zero_sales_count/total_records*100:.1f}%)")

promo_pct = (total_on_promo/total_records*100) if total_records > 0 else 0
print(f"\n🎯 PROMOTION:")
print(f"  Promotional items: {total_on_promo:,} ({promo_pct:.1f}%)")

print(f"\n⏱️ PERFORMANCE:")
print(f"  Total time: {load_time:.1f}s")
print(f"  Chunks processed: {chunks_processed}")
print(f"  Speed: {total_records/load_time:.0f} rows/second")

# ============================================
# STEP 4: LOAD SAMPLE FOR DETAILED ANALYSIS
# ============================================

print("\n" + "="*60)
print("LOADING SAMPLE FOR DETAILED ANALYSIS")
print("="*60)

print(f"\nLoading first 500k records for EDA...")
sample_start = time.time()

df_sample = pd.read_csv(
    csv_path,
    nrows=500_000,
    dtype={
        'store_nbr': 'uint16',
        'item_nbr': 'uint32',
        'unit_sales': 'float32',
        'onpromotion': 'float32'
    }
)

df_sample['date'] = pd.to_datetime(df_sample['date'])

sample_time = time.time() - sample_start
print(f"✓ Loaded {len(df_sample):,} rows in {sample_time:.1f}s")

print(f"\nSample data shape: {df_sample.shape}")
print(f"\nFirst 10 rows:")
print(df_sample.head(10))

print(f"\nData types:")
print(df_sample.dtypes)

print(f"\nNull values:")
print(df_sample.isnull().sum())

# ============================================
# STEP 5: BASIC DISTRIBUTIONS
# ============================================

print("\n" + "="*60)
print("SAMPLE DISTRIBUTIONS")
print("="*60)

print(f"\nUnique stores in sample: {df_sample['store_nbr'].nunique()}")
print(f"Unique items in sample: {df_sample['item_nbr'].nunique()}")

print(f"\nTop 10 stores by total sales:")
top_stores = df_sample.groupby('store_nbr')['unit_sales'].sum().nlargest(10)
print(top_stores)

print(f"\nTop 10 items by total sales:")
top_items = df_sample.groupby('item_nbr')['unit_sales'].sum().nlargest(10)
print(top_items)

# ============================================
# STEP 6: READY FOR NEXT STEPS
# ============================================

print("\n" + "="*60)
print("✓ EDA COMPLETE - READY FOR MODELING")
print("="*60)

print(f"""
SUMMARY:
✓ Total dataset: {total_records:,} records
✓ Time series: {(max_date - min_date).days} days
✓ Store-Item combinations: {len(stores_seen)} stores × {len(items_seen):,} items
✓ Sparsity: {zero_sales_count/total_records*100:.1f}% zero sales
✓ Promotion rate: {promo_pct:.1f}%

NEXT STEPS:
1. Load stores.csv for demographics
2. Load items.csv for product categories
3. Load holidays_events.csv for events/seasonality
4. Merge all datasets on keys: date + store + item
5. Engineer features: time-based + demographics
6. Build demand prediction model

LOADING TIME: {load_time:.1f}s (VERY FAST!)
""")

print(f"Total execution: {time.time() - start:.1f}s")


🚀 OPTIMIZED STREAMING EDA - CORPORACIÓN FAVORITA

File size: 4.65 GB
Using optimized chunked streaming...

STREAMING ANALYSIS (Processing in chunks)

Columns to process:
  date_col: date
  store_col: store_nbr
  item_col: item_nbr
  demand_col: unit_sales
  promo_col: onpromotion
  id_col: id

Processing chunks of 100,000 rows...

  ✓ 1,000,000 records (0.3s, 3306270 rows/sec)
  ✓ 2,000,000 records (0.6s, 3438666 rows/sec)
  ✓ 3,000,000 records (0.9s, 3341912 rows/sec)
  ✓ 4,000,000 records (1.2s, 3270620 rows/sec)
  ✓ 5,000,000 records (1.5s, 3307618 rows/sec)
  ✓ 6,000,000 records (1.8s, 3329045 rows/sec)
  ✓ 7,000,000 records (2.1s, 3305846 rows/sec)
  ✓ 8,000,000 records (2.4s, 3289075 rows/sec)
  ✓ 9,000,000 records (2.7s, 3303127 rows/sec)
  ✓ 10,000,000 records (3.0s, 3309611 rows/sec)
  ✓ 11,000,000 records (3.3s, 3339311 rows/sec)
  ✓ 12,000,000 records (3.6s, 3357621 rows/sec)
  ✓ 13,000,000 records (3.8s, 3378082 rows/sec)
  ✓ 14,000,000 records (4.1s, 3396625 rows/sec)
  ✓ 

In [6]:
import pandas as pd
import numpy as np
import time

print("="*60)
print("PHASE 2: MERGE DEMOGRAPHICS & EVENTS")
print("="*60)

# ============================================
# STEP 1: LOAD STORES (DEMOGRAPHICS)
# ============================================

print("\n1️⃣ Loading stores.csv (Demographics)...")
start = time.time()

stores = pd.read_csv('/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/stores.csv')

print(f"✓ Loaded {len(stores)} stores in {time.time()-start:.2f}s")
print(f"\nStores data:")
print(stores.head())
print(f"\nStores info:")
print(stores.info())

# ============================================
# STEP 2: LOAD ITEMS (PRODUCT CATEGORIES)
# ============================================

print("\n2️⃣ Loading items.csv (Product Categories)...")
start = time.time()

items = pd.read_csv('/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/items.csv')

print(f"✓ Loaded {len(items)} items in {time.time()-start:.2f}s")
print(f"\nItems data:")
print(items.head())
print(f"\nItems info:")
print(items.info())

# ============================================
# STEP 3: LOAD HOLIDAYS & EVENTS
# ============================================

print("\n3️⃣ Loading holidays_events.csv (Seasonality)...")
start = time.time()

holidays = pd.read_csv('/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/holidays_events.csv',
                        parse_dates=['date'])

print(f"✓ Loaded {len(holidays)} holiday events in {time.time()-start:.2f}s")
print(f"\nHolidays data:")
print(holidays.head())
print(f"\nHoliday types:")
print(holidays['type'].value_counts())

# ============================================
# STEP 4: LOAD OIL PRICES (EXTERNAL FACTOR)
# ============================================

print("\n4️⃣ Loading oil.csv (Economic Factor)...")
start = time.time()

oil = pd.read_csv('/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/oil.csv',
                   parse_dates=['date'])

print(f"✓ Loaded {len(oil)} oil price records in {time.time()-start:.2f}s")
print(f"\nOil price stats:")
print(oil['dcoilwtico'].describe())

# ============================================
# STEP 5: LOAD TRANSACTIONS (CONTEXT)
# ============================================

print("\n5️⃣ Loading transactions.csv (Store Activity)...")
start = time.time()

transactions = pd.read_csv('/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/transactions.csv',
                           parse_dates=['date'])

print(f"✓ Loaded {len(transactions)} transaction records in {time.time()-start:.2f}s")
print(f"\nTransactions stats:")
print(transactions.head())

# ============================================
# STEP 6: LOAD SAMPLE DATA & MERGE
# ============================================

print("\n" + "="*60)
print("MERGING ALL DATASETS")
print("="*60)

print("\n6️⃣ Loading sales sample (first 1M records)...")
start = time.time()

df = pd.read_csv(
    '/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/train.csv',
    nrows=1_000_000,
    dtype={'store_nbr': 'uint16', 'item_nbr': 'uint32', 'unit_sales': 'float32', 'onpromotion': 'float32'}
)
df['date'] = pd.to_datetime(df['date'])

print(f"✓ Loaded {len(df):,} sales records in {time.time()-start:.2f}s")

print("\n7️⃣ Merging with stores (demographics)...")
start = time.time()

df = df.merge(stores, on='store_nbr', how='left')
print(f"✓ Merged in {time.time()-start:.2f}s")
print(f"  Shape: {df.shape}")
print(f"  New columns: {stores.columns.tolist()}")

print("\n8️⃣ Merging with items (product categories)...")
start = time.time()

df = df.merge(items, on='item_nbr', how='left')
print(f"✓ Merged in {time.time()-start:.2f}s")
print(f"  Shape: {df.shape}")
print(f"  New columns: {items.columns.tolist()}")

print("\n9️⃣ Merging with holidays (events/seasonality)...")
start = time.time()

df = df.merge(holidays, on='date', how='left')
print(f"✓ Merged in {time.time()-start:.2f}s")
print(f"  Shape: {df.shape}")
print(f"  Holiday columns: {holidays.columns.tolist()}")

print("\n🔟 Merging with oil prices (economic factor)...")
start = time.time()

df = df.merge(oil, on='date', how='left')
print(f"✓ Merged in {time.time()-start:.2f}s")
print(f"  Shape: {df.shape}")

print("\n1️⃣1️⃣ Merging with transactions (store activity)...")
start = time.time()

df = df.merge(transactions, on=['date', 'store_nbr'], how='left')
print(f"✓ Merged in {time.time()-start:.2f}s")
print(f"  Shape: {df.shape}")

# ============================================
# STEP 7: FINAL MERGED DATASET
# ============================================

print("\n" + "="*60)
print("FINAL MERGED DATASET")
print("="*60)

print(f"\nShape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}")

print(f"\nData types:")
print(df.dtypes)

print(f"\nNull values (check for merge issues):")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

print(f"\nFirst 10 rows (merged data):")
print(df.head(10))

# ============================================
# STEP 8: BASIC EDA ON MERGED DATA
# ============================================

print("\n" + "="*60)
print("MERGED DATA ANALYSIS")
print("="*60)

print(f"\n📊 Demographics breakdown:")
print(f"\nStore types:")
print(df['type_x'].value_counts())

print(f"\nStore clusters:")
print(df['cluster'].value_counts())

print(f"\nStore cities (top 10):")
print(df['city'].value_counts().head(10))

print(f"\n📦 Product categories:")
print(df['family'].value_counts().head(10))

print(f"\n🎯 Holiday events in data:")
print(df['type_x'].value_counts() if 'type_x' in df.columns else "No holiday events in this sample")

print(f"\n💰 Sales by store type:")
print(df.groupby('type_x')['unit_sales'].agg(['sum', 'mean', 'count']))

# ============================================
# STEP 9: SAVE FOR MODELING
# ============================================

print("\n" + "="*60)
print("SAVING MERGED DATASET")
print("="*60)

df.to_csv('favorita_merged_sample.csv', index=False)
print(f"\n✓ Saved to: favorita_merged_sample.csv ({len(df):,} rows)")

print(f"""
✓ READY FOR FEATURE ENGINEERING & MODELING

NEXT STEPS:
1. Engineer time-based features (day, week, month, seasonality)
2. Create lag features for time series (t-1, t-7, t-30)
3. One-hot encode categorical features (store type, family, etc.)
4. Handle negative sales (adjust or remove outliers)
5. Handle missing onpromotion values (backfill or mark)
6. Train demand forecasting model
7. Evaluate with time series cross-validation

RECOMMENDED MODELS:
- LightGBM (fast, handles categorical, non-linear)
- XGBoost (robust, good for time series with exogenous vars)
- SARIMA/Prophet (if you want statistical time series)
- LSTM/Neural Networks (if you want deep learning)
""")


PHASE 2: MERGE DEMOGRAPHICS & EVENTS

1️⃣ Loading stores.csv (Demographics)...
✓ Loaded 54 stores in 0.01s

Stores data:
   store_nbr           city                           state type  cluster
0          1          Quito                       Pichincha    D       13
1          2          Quito                       Pichincha    D       13
2          3          Quito                       Pichincha    D        8
3          4          Quito                       Pichincha    D        9
4          5  Santo Domingo  Santo Domingo de los Tsachilas    D        4

Stores info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+

In [1]:
import pandas as pd
import numpy as np
import time

print("="*60)
print("PHASE 1: LOAD AUXILIARY DATA & DEFINE TARGET")
print("="*60)

# Define file paths
BASE_PATH = '/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/'
MAIN_TRAIN_FILE = f"{BASE_PATH}train.csv"
OUTPUT_FILE = "grocery_i_sales_complete.csv"

# Define our target product category
TARGET_FAMILY = 'GROCERY I'

start_total = time.time()

# 1. Load Items to find our target item numbers
print(f"Loading items.csv to filter for '{TARGET_FAMILY}'...")
items = pd.read_csv(f"{BASE_PATH}items.csv")
grocery_items = items[items['family'] == TARGET_FAMILY]
grocery_item_numbers = set(grocery_items['item_nbr'])
print(f"✓ Found {len(grocery_item_numbers):,} items in '{TARGET_FAMILY}' category.\n")

# 2. Load Stores
print("Loading stores.csv (Demographics)...")
stores = pd.read_csv(f"{BASE_PATH}stores.csv")
print(f"✓ Loaded {len(stores)} stores.\n")

# 3. Load & Pre-process Oil
print("Loading oil.csv (Economic Factor)...")
oil = pd.read_csv(f"{BASE_PATH}oil.csv", parse_dates=['date'])
# Oil data has gaps for weekends/holidays. Forward-fill to impute missing values.
oil = oil.set_index('date').resample('D').ffill().reset_index()
oil['dcoilwtico'] = oil['dcoilwtico'].fillna(method='ffill')
print(f"✓ Loaded and forward-filled oil prices.\n")

# 4. Load & Pre-process Holidays
print("Loading holidays_events.csv (Seasonality)...")
holidays = pd.read_csv(f"{BASE_PATH}holidays_events.csv", parse_dates=['date'])
# Simplify: We'll focus on national holidays/events for this broad category
holidays_national = holidays[holidays['locale'] == 'National'].copy()
holidays_national.rename(columns={'type': 'holiday_type'}, inplace=True)
print(f"✓ Loaded and filtered for {len(holidays_national)} national holidays/events.\n")

# 5. Load Transactions
print("Loading transactions.csv (Store Activity)...")
transactions = pd.read_csv(f"{BASE_PATH}transactions.csv", parse_dates=['date'])
print(f"✓ Loaded {len(transactions):,} transaction records.\n")


print("="*60)
print(f"PHASE 2: STREAMING & FILTERING '{TARGET_FAMILY}' DATA")
print("="*60)

chunk_size = 5_000_000  # Process 5 million rows at a time
chunks_processed = 0
all_filtered_chunks = []

print(f"Streaming {MAIN_TRAIN_FILE} in {chunk_size:,} row chunks...")
start_stream = time.time()

# Read the massive train.csv file in chunks
for chunk in pd.read_csv(
    MAIN_TRAIN_FILE,
    chunksize=chunk_size,
    dtype={'store_nbr': 'uint16', 'item_nbr': 'uint32', 'unit_sales': 'float32', 'onpromotion': 'float32'}
):
    chunks_processed += 1
    
    # Filter the chunk to keep only rows where item_nbr is in our target set
    filtered_chunk = chunk[chunk['item_nbr'].isin(grocery_item_numbers)]
    
    if not filtered_chunk.empty:
        all_filtered_chunks.append(filtered_chunk)
        
    print(f"  ✓ Chunk {chunks_processed}: Found {len(filtered_chunk):,} '{TARGET_FAMILY}' records.")

# Concatenate all the filtered chunks into one DataFrame
print("\nCombining all filtered chunks...")
df = pd.concat(all_filtered_chunks, ignore_index=True)
df['date'] = pd.to_datetime(df['date'])

stream_time = time.time() - start_stream
print(f"✓ Streaming complete in {stream_time:.2f}s.")
print(f"  Total records found for '{TARGET_FAMILY}': {len(df):,}")
print(f"  Original size (approx): 125.5M rows. New size: {len(df):,} rows.")
print(f"  Data reduction: {1 - len(df)/125497040:.1%}\n")


print("="*60)
print("PHASE 3: MERGING ALL DATASETS")
print("="*60)
start_merge = time.time()

# 1. Merge with Stores (Demographics)
print(f"Merging with stores...")
df = df.merge(stores, on='store_nbr', how='left')

# 2. Merge with Items (Product Categories)
print(f"Merging with items...")
df = df.merge(grocery_items, on='item_nbr', how='left') # Use the filtered items list

# 3. Merge with Oil (Economic Factor)
print(f"Merging with oil prices...")
df = df.merge(oil, on='date', how='left')

# 4. Merge with Holidays (Events/Seasonality)
print(f"Merging with national holidays...")
df = df.merge(holidays_national, on='date', how='left')

# 5. Merge with Transactions (Store Activity)
print(f"Merging with transactions...")
df = df.merge(transactions, on=['date', 'store_nbr'], how='left')

merge_time = time.time() - start_merge
print(f"✓ All merges complete in {merge_time:.2f}s.\n")


print("="*60)
print("PHASE 4: FINAL DATASET OVERVIEW & SAVE")
print("="*60)

print(f"Final merged dataset shape: {df.shape}")
print(f"Final memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\nFinal columns:")
print(df.columns.tolist())

print("\nNull values (top 10):")
print(df.isnull().sum().nlargest(10))

print("\nData sample (first 5 rows):")
print(df.head())

# Save the final, analysis-ready dataset
print(f"\nSaving to '{OUTPUT_FILE}'...")
df.to_csv(OUTPUT_FILE, index=False)

total_time = time.time() - start_total
print(f"\n✓ SUCCESS! Saved {len(df):,} rows to '{OUTPUT_FILE}'.")
print(f"Total time: {total_time:.2f}s")
print("\nThis dataset is now ready for preprocessing and feature engineering.")

/Users/shashwatkushwaha/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


PHASE 1: LOAD AUXILIARY DATA & DEFINE TARGET
Loading items.csv to filter for 'GROCERY I'...
✓ Found 1,334 items in 'GROCERY I' category.

Loading stores.csv (Demographics)...
✓ Loaded 54 stores.

Loading oil.csv (Economic Factor)...
✓ Loaded and forward-filled oil prices.

Loading holidays_events.csv (Seasonality)...
✓ Loaded and filtered for 174 national holidays/events.

Loading transactions.csv (Store Activity)...
✓ Loaded 83,488 transaction records.

PHASE 2: STREAMING & FILTERING 'GROCERY I' DATA
Streaming /Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/train.csv in 5,000,000 row chunks...


/var/folders/11/cfwlhk514vg3gv00qsfbjx8r0000gn/T/ipykernel_44797/1705793821.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  oil['dcoilwtico'] = oil['dcoilwtico'].fillna(method='ffill')


  ✓ Chunk 1: Found 2,148,790 'GROCERY I' records.
  ✓ Chunk 2: Found 2,186,412 'GROCERY I' records.
  ✓ Chunk 3: Found 2,159,241 'GROCERY I' records.
  ✓ Chunk 4: Found 1,858,202 'GROCERY I' records.
  ✓ Chunk 5: Found 1,919,952 'GROCERY I' records.
  ✓ Chunk 6: Found 1,878,731 'GROCERY I' records.
  ✓ Chunk 7: Found 1,622,510 'GROCERY I' records.
  ✓ Chunk 8: Found 1,767,442 'GROCERY I' records.
  ✓ Chunk 9: Found 2,058,225 'GROCERY I' records.
  ✓ Chunk 10: Found 1,836,518 'GROCERY I' records.
  ✓ Chunk 11: Found 1,675,833 'GROCERY I' records.
  ✓ Chunk 12: Found 1,680,973 'GROCERY I' records.
  ✓ Chunk 13: Found 1,748,719 'GROCERY I' records.
  ✓ Chunk 14: Found 1,751,401 'GROCERY I' records.
  ✓ Chunk 15: Found 1,765,901 'GROCERY I' records.
  ✓ Chunk 16: Found 1,754,621 'GROCERY I' records.
  ✓ Chunk 17: Found 1,778,313 'GROCERY I' records.
  ✓ Chunk 18: Found 1,738,168 'GROCERY I' records.
  ✓ Chunk 19: Found 1,731,918 'GROCERY I' records.
  ✓ Chunk 20: Found 1,730,616 'GROCERY I

In [2]:
import pandas as pd
import numpy as np
import time

print("="*60)
print("PHASE 1: LOAD AUXILIARY DATA & DEFINE TARGET")
print("="*60)

# --- Configuration ---
BASE_PATH = '/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/'
MAIN_TRAIN_FILE = f"{BASE_PATH}train.csv"
HERO_ITEM_NBR = 807493  # Selected from your EDA of top-selling items
OUTPUT_FILE = f"hero_item_{HERO_ITEM_NBR}_complete.csv"

print(f"Targeting Hero Item: {HERO_ITEM_NBR}")
start_total = time.time()

# --- 1. Load Auxiliary Data ---
print("\nLoading auxiliary datasets...")
stores = pd.read_csv(f"{BASE_PATH}stores.csv")
items = pd.read_csv(f"{BASE_PATH}items.csv")
holidays = pd.read_csv(f"{BASE_PATH}holidays_events.csv", parse_dates=['date'])
oil = pd.read_csv(f"{BASE_PATH}oil.csv", parse_dates=['date'])
transactions = pd.read_csv(f"{BASE_PATH}transactions.csv", parse_dates=['date'])
print("✓ Auxiliary data loaded.")

# --- 2. Pre-process Auxiliary Data ---
print("Pre-processing auxiliary data...")
# Pre-process Oil: Forward-fill missing values (weekends/holidays)
oil = oil.set_index('date').resample('D').ffill().reset_index()
oil['dcoilwtico'] = oil['dcoilwtico'].fillna(method='ffill')

# Pre-process Holidays: Rename 'type' to avoid collisions
holidays.rename(columns={'type': 'holiday_type'}, inplace=True)
print("✓ Auxiliary data pre-processed.\n")


print("="*60)
print(f"PHASE 2: STREAMING & FILTERING FOR ITEM {HERO_ITEM_NBR}")
print("="*60)

chunk_size = 5_000_000
chunks_processed = 0
all_filtered_chunks = []

print(f"Streaming {MAIN_TRAIN_FILE} in {chunk_size:,} row chunks...")
start_stream = time.time()

# Read the massive train.csv file in chunks
for chunk in pd.read_csv(
    MAIN_TRAIN_FILE,
    chunksize=chunk_size,
    dtype={'store_nbr': 'uint16', 'item_nbr': 'uint32', 'unit_sales': 'float32', 'onpromotion': 'float32'}
):
    chunks_processed += 1
    
    # Filter the chunk to keep only rows for our HERO_ITEM_NBR
    filtered_chunk = chunk[chunk['item_nbr'] == HERO_ITEM_NBR]
    
    if not filtered_chunk.empty:
        all_filtered_chunks.append(filtered_chunk)
        
    print(f"  ✓ Chunk {chunks_processed}: Found {len(filtered_chunk):,} records for our item.")

# Concatenate all the filtered chunks into one DataFrame
print("\nCombining all filtered chunks...")
df = pd.concat(all_filtered_chunks, ignore_index=True)
df['date'] = pd.to_datetime(df['date'])

stream_time = time.time() - start_stream
print(f"✓ Streaming complete in {stream_time:.2f}s.")
print(f"  Total records found for item {HERO_ITEM_NBR}: {len(df):,}")
print(f"  This dataset has the FULL 5-year time series.\n")


print("="*60)
print("PHASE 3: MERGING ALL CONTEXTUAL DATA")
print("="*60)
start_merge = time.time()

# 1. Merge with Stores (Demographics)
print(f"Merging with stores...")
df = df.merge(stores, on='store_nbr', how='left')

# 2. Merge with Items (Product Categories)
print(f"Merging with items...")
# We can merge with the full 'items' df, but filtering first is slightly more efficient
hero_item_info = items[items['item_nbr'] == HERO_ITEM_NBR]
df = df.merge(hero_item_info, on='item_nbr', how='left')

# 3. Merge with Oil (Economic Factor)
print(f"Merging with oil prices...")
df = df.merge(oil, on='date', how='left')

# 4. Merge with Holidays (Events/Seasonality)
print(f"Merging with holidays...")
# We use a left merge to keep all sales days and get nulls for non-holidays
df = df.merge(holidays, on='date', how='left')

# 5. Merge with Transactions (Store Activity)
print(f"Merging with transactions...")
df = df.merge(transactions, on=['date', 'store_nbr'], how='left')

merge_time = time.time() - start_merge
print(f"✓ All merges complete in {merge_time:.2f}s.\n")


print("="*60)
print("PHASE 4: FINAL DATASET OVERVIEW & SAVE")
print("="*60)

print(f"Final merged dataset shape: {df.shape}")
print(f"Final memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\nFinal columns:")
print(df.columns.tolist())

print("\nNull values (top 10):")
print(df.isnull().sum().nlargest(10))

print("\nPromotion data availability:")
print(df['onpromotion'].value_counts(dropna=False))

print("\nData sample (first 5 rows):")
print(df.head())

# Save the final, analysis-ready dataset
print(f"\nSaving to '{OUTPUT_FILE}'...")
df.to_csv(OUTPUT_FILE, index=False)

total_time = time.time() - start_total
print(f"\n✓ SUCCESS! Saved {len(df):,} rows to '{OUTPUT_FILE}'.")
print(f"Total time: {total_time:.2f}s")
print("\nThis single-item dataset is now ready for preprocessing and feature engineering.")

PHASE 1: LOAD AUXILIARY DATA & DEFINE TARGET
Targeting Hero Item: 807493

Loading auxiliary datasets...
✓ Auxiliary data loaded.
Pre-processing auxiliary data...
✓ Auxiliary data pre-processed.

PHASE 2: STREAMING & FILTERING FOR ITEM 807493


/var/folders/11/cfwlhk514vg3gv00qsfbjx8r0000gn/T/ipykernel_44797/4048654883.py:31: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  oil['dcoilwtico'] = oil['dcoilwtico'].fillna(method='ffill')


Streaming /Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/datasets/Ecuador Data/favorita-grocery-sales-forecasting/train.csv in 5,000,000 row chunks...
  ✓ Chunk 1: Found 5,412 records for our item.
  ✓ Chunk 2: Found 5,153 records for our item.
  ✓ Chunk 3: Found 4,958 records for our item.
  ✓ Chunk 4: Found 4,135 records for our item.
  ✓ Chunk 5: Found 4,291 records for our item.
  ✓ Chunk 6: Found 4,039 records for our item.
  ✓ Chunk 7: Found 3,402 records for our item.
  ✓ Chunk 8: Found 3,507 records for our item.
  ✓ Chunk 9: Found 3,976 records for our item.
  ✓ Chunk 10: Found 3,431 records for our item.
  ✓ Chunk 11: Found 3,029 records for our item.
  ✓ Chunk 12: Found 2,866 records for our item.
  ✓ Chunk 13: Found 2,796 records for our item.
  ✓ Chunk 14: Found 2,728 records for our item.
  ✓ Chunk 15: Found 2,699 records for our item.
  ✓ Chunk 16: Found 2,554 records for our item.
  ✓ Chunk 17: Found 1,532 records for our item.
  ✓ Chunk 18: Found 2,5

In [3]:
import pandas as pd
import numpy as np
import time

print("="*60)
print("PHASE 1: LOAD DATA TO BE CLEANED")
print("="*60)

INPUT_FILE = "hero_item_807493_complete.csv"
OUTPUT_FILE = f"hero_item_807493_preprocessed.csv"

start_total = time.time()

print(f"Loading '{INPUT_FILE}'...")
try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"ERROR: File not found. Please run the 'extract_hero_item.py' script first.")
    exit()

print(f"✓ Loaded {len(df):,} rows.")
print(f"  Initial memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"  Initial null values:\n{df.isnull().sum().nlargest(10)}\n")


print("="*60)
print("PHASE 2: RESOLVING DATA QUALITY FLAGS")
print("="*60)

# --- 1. Resolve Flag 1: Duplicate Rows from Holiday Merge ---
print(f"Resolving Flag 1: Duplicate Rows...")
initial_rows = len(df)
df.drop_duplicates(subset=['id'], keep='first', inplace=True)
final_rows = len(df)
print(f"✓ Dropped {initial_rows - final_rows:,} duplicate rows. New shape: {df.shape}\n")

# --- 2. Resolve Flag 2: Missing 'onpromotion' Data ---
print(f"Resolving Flag 2: 'onpromotion' nulls...")
onpromotion_nulls = df['onpromotion'].isnull().sum()
# Fill NaN with 0 (assuming not-tracked means not-on-promotion)
df['onpromotion'] = df['onpromotion'].fillna(0).astype('uint8')
print(f"✓ Filled {onpromotion_nulls:,} 'onpromotion' nulls with 0.\n")

# --- 3. Resolve Flag 3: Holiday Data & Nulls ---
print(f"Resolving Flag 3: Holiday columns...")
# Create a simple binary flag
df['is_holiday'] = (~df['holiday_type'].isnull()).astype('uint8')
print(f"✓ Created 'is_holiday' flag. (Found {df['is_holiday'].sum():,} holiday-related days)")

# Fill remaining holiday text columns with a placeholder
holiday_text_cols = ['holiday_type', 'locale', 'locale_name', 'description']
df[holiday_text_cols] = df[holiday_text_cols].fillna("NoHoliday")
# Drop the 'transferred' column as it's less useful for this model
if 'transferred' in df.columns:
    df.drop('transferred', axis=1, inplace=True)
print(f"✓ Cleaned and simplified holiday feature columns.\n")

# --- 4. Resolve Flag 4: Minor Nulls (Oil & Transactions) ---
print(f"Resolving Flag 4: Minor nulls...")
# Forward-fill the few oil price nulls
oil_nulls = df['dcoilwtico'].isnull().sum()
df['dcoilwtico'] = df['dcoilwtico'].fillna(method='ffill')
# Backward-fill in case the very first value was null
df['dcoilwtico'] = df['dcoilwtico'].fillna(method='bfill')
print(f"✓ Imputed {oil_nulls} 'dcoilwtico' nulls using ffill/bfill.")

# Fill missing transactions with the store's average
trans_nulls = df['transactions'].isnull().sum()
df['transactions'] = df['transactions'].fillna(df.groupby('store_nbr')['transactions'].transform('mean'))
# Fill any remaining (for stores with 0 non-nulls) with the global mean
df['transactions'] = df['transactions'].fillna(df['transactions'].mean())
print(f"✓ Imputed {trans_nulls} 'transactions' nulls using store/global mean.\n")


print("="*60)
print("PHASE 3: FINAL CHECK & SAVE")
print("="*60)

print(f"Final dataset shape: {df.shape}")
print(f"Final memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\nFinal null value check (should all be 0):")
print(df.isnull().sum())

# Save the final, analysis-ready dataset
print(f"\nSaving to '{OUTPUT_FILE}'...")
df.to_csv(OUTPUT_FILE, index=False)

total_time = time.time() - start_total
print(f"\n✓ SUCCESS! Saved {len(df):,} preprocessed rows to '{OUTPUT_FILE}'.")
print(f"Total time: {total_time:.2f}s")
print("\nThis dataset is now clean and ready for feature engineering.")

PHASE 1: LOAD DATA TO BE CLEANED
Loading 'hero_item_807493_complete.csv'...
✓ Loaded 81,363 rows.
  Initial memory usage: 45.4 MB
  Initial null values:
holiday_type    67962
locale          67962
locale_name     67962
description     67962
transferred     67962
onpromotion     21088
transactions      117
dcoilwtico          1
id                  0
date                0
dtype: int64

PHASE 2: RESOLVING DATA QUALITY FLAGS
Resolving Flag 1: Duplicate Rows...
✓ Dropped 1,478 duplicate rows. New shape: (79885, 20)

Resolving Flag 2: 'onpromotion' nulls...
✓ Filled 20,855 'onpromotion' nulls with 0.

Resolving Flag 3: Holiday columns...
✓ Created 'is_holiday' flag. (Found 11,923 holiday-related days)
✓ Cleaned and simplified holiday feature columns.

Resolving Flag 4: Minor nulls...
✓ Imputed 1 'dcoilwtico' nulls using ffill/bfill.
✓ Imputed 117 'transactions' nulls using store/global mean.

PHASE 3: FINAL CHECK & SAVE
Final dataset shape: (79885, 20)
Final memory usage: 50.9 MB

Final null

/var/folders/11/cfwlhk514vg3gv00qsfbjx8r0000gn/T/ipykernel_44797/1798542140.py:62: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['dcoilwtico'] = df['dcoilwtico'].fillna(method='ffill')
/var/folders/11/cfwlhk514vg3gv00qsfbjx8r0000gn/T/ipykernel_44797/1798542140.py:64: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['dcoilwtico'] = df['dcoilwtico'].fillna(method='bfill')



✓ SUCCESS! Saved 79,885 preprocessed rows to 'hero_item_807493_preprocessed.csv'.
Total time: 0.55s

This dataset is now clean and ready for feature engineering.
